In [129]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
import ast

# Locate the data folder
data_folder = Path("../data")

print("Files in the data folder:")

for file in data_folder.iterdir():
    print(file.name)

Files in the data folder:
goodreads_data.csv
README.md


In [130]:
# Load the Goodreads dataset
df = pd.read_csv("../data/goodreads_data.csv")

# Display the first 5 rows
df.head()

,Unnamed: 0,Book,Author,Description,Genres,Avg_Rating,Num_Ratings,URL
0,0,To Kill a Mockingbird,Harper Lee,The unforgettable novel of a childhood in a sl...,"['Classics', 'Fiction', 'Historical Fiction', ...",4.27,"5,691,311",https://www.goodreads.com/book/show/2657.To_Ki...
1,1,Harry Potter and the Philosopher’s Stone (Harr...,J.K. Rowling,Harry Potter thinks he is an ordinary boy - un...,"['Fantasy', 'Fiction', 'Young Adult', 'Magic',...",4.47,"9,278,135",https://www.goodreads.com/book/show/72193.Harr...
2,2,Pride and Prejudice,Jane Austen,"Since its immediate success in 1813, Pride and...","['Classics', 'Fiction', 'Romance', 'Historical...",4.28,"3,944,155",https://www.goodreads.com/book/show/1885.Pride...
3,3,The Diary of a Young Girl,Anne Frank,Discovered in the attic in which she spent the...,"['Classics', 'Nonfiction', 'History', 'Biograp...",4.18,"3,488,438",https://www.goodreads.com/book/show/48855.The_...
4,4,Animal Farm,George Orwell,Librarian's note: There is an Alternate Cover ...,"['Classics', 'Fiction', 'Dystopia', 'Fantasy',...",3.98,"3,575,172",https://www.goodreads.com/book/show/170448.Ani...


In [131]:
# Display the number of rows and columns
print("Dataset dimensions:", df.shape)

# Display all column names
print("\nColumn names:")
print(df.columns.tolist())

# Display information about each column
df.info()

Dataset dimensions: (10000, 8)

Column names:
['Unnamed: 0', 'Book', 'Author', 'Description', 'Genres', 'Avg_Rating', 'Num_Ratings', 'URL']
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   10000 non-null  int64  
 1   Book         10000 non-null  str    
 2   Author       10000 non-null  str    
 3   Description  9923 non-null   str    
 4   Genres       10000 non-null  str    
 5   Avg_Rating   10000 non-null  float64
 6   Num_Ratings  10000 non-null  str    
 7   URL          10000 non-null  str    
dtypes: float64(1), int64(1), str(6)
memory usage: 625.1 KB


In [132]:
for genre in df["Genres"].head(10):
    print(genre)

['Classics', 'Fiction', 'Historical Fiction', 'School', 'Literature', 'Young Adult', 'Historical']
['Fantasy', 'Fiction', 'Young Adult', 'Magic', 'Childrens', 'Middle Grade', 'Classics']
['Classics', 'Fiction', 'Romance', 'Historical Fiction', 'Literature', 'Historical', 'Audiobook']
['Classics', 'Nonfiction', 'History', 'Biography', 'Memoir', 'Historical', 'Holocaust']
['Classics', 'Fiction', 'Dystopia', 'Fantasy', 'Politics', 'School', 'Literature']
['Classics', 'Fiction', 'Fantasy', 'Childrens', 'France', 'Philosophy', 'Young Adult']
['Classics', 'Fiction', 'Science Fiction', 'Dystopia', 'Literature', 'Novels', 'Politics']
['Classics', 'Fiction', 'School', 'Historical Fiction', 'Literature', 'Romance', 'Novels']
['Classics', 'Fiction', 'Young Adult', 'Literature', 'School', 'Novels', 'Coming Of Age']
['Fantasy', 'Classics', 'Fiction', 'Adventure', 'Science Fiction Fantasy', 'Epic Fantasy', 'High Fantasy']


In [133]:
# Count how many books have missing descriptions
print("Missing descriptions:", df["Description"].isna().sum())

# Check for duplicate books
print("Duplicate book titles:", df["Book"].duplicated().sum())

# Check the number of unique genre entries
print("Unique genre combinations:", df["Genres"].nunique())

# Find books with missing descriptions
missing_books = df[df["Description"].isna()]

# Display their titles and URLs
missing_books[["Book", "Author", "URL"]].head(20)

Missing descriptions: 77
Duplicate book titles: 129
Unique genre combinations: 8043


,Book,Author,URL
1066,A Decade of Desire: Erotic Tales from the Char...,Charles Dyson,https://www.goodreads.com/book/show/57601103-a...
1224,"Complicated Moonlight (DCYE, #2)",Lynessa Layne,https://www.goodreads.com/book/show/55196363-c...
1745,"Mad Love (DCYE, #3)",Lynessa Layne,https://www.goodreads.com/book/show/55961601-m...
2104,The Spirit of Prayer: The Believer's Authority...,Tolulope Oyewole,https://www.goodreads.com/book/show/57392898-t...
2270,Lift: Five Practices Great Managers Do Consist...,Raymond Wheeler,https://www.goodreads.com/book/show/55455888-lift
2279,Glucose Control Eating: Lose Weight Stay Slimm...,Rick Mystrom,https://www.goodreads.com/book/show/65498175-g...
2362,SHADOW PANTHEON,Eric Nierstedt,https://www.goodreads.com/book/show/53935101-s...
2728,Counter Identity (Remmich/Miller #2),Karl Braungart,https://www.goodreads.com/book/show/61350389-c...
2845,Dear Su Yen: A Young Woman From Taiwan Discove...,Su-Yen Hu,https://www.goodreads.com/book/show/9698540-de...
2850,Sex: How to Get More of It: A guy's roadmap to...,Jodi Ambrose,https://www.goodreads.com/book/show/10524630-sex


In [134]:
# Create a working copy
df_clean = df.copy()

# Remove the unnecessary index column
df_clean = df_clean.drop(columns=["Unnamed: 0"])

df_clean["Genre_List"] = df_clean["Genres"].apply(ast.literal_eval)

# Keep only books that have at least one genre
df_clean = df_clean[
    df_clean["Genre_List"].apply(len) > 0
].copy()

# Use the first listed genre as our primary genre
df_clean["Primary_Genre"] = df_clean["Genre_List"].apply(
    lambda genres: genres[0]
)

# Display the first five rows
df_clean[["Book", "Description", "Genre_List", "Primary_Genre"]].head()

,Book,Description,Genre_List,Primary_Genre
0,To Kill a Mockingbird,The unforgettable novel of a childhood in a sl...,"[Classics, Fiction, Historical Fiction, School...",Classics
1,Harry Potter and the Philosopher’s Stone (Harr...,Harry Potter thinks he is an ordinary boy - un...,"[Fantasy, Fiction, Young Adult, Magic, Childre...",Fantasy
2,Pride and Prejudice,"Since its immediate success in 1813, Pride and...","[Classics, Fiction, Romance, Historical Fictio...",Classics
3,The Diary of a Young Girl,Discovered in the attic in which she spent the...,"[Classics, Nonfiction, History, Biography, Mem...",Classics
4,Animal Farm,Librarian's note: There is an Alternate Cover ...,"[Classics, Fiction, Dystopia, Fantasy, Politic...",Classics


In [135]:
# Find books that have empty genre lists
empty_genres = df_clean[df_clean["Genre_List"].apply(len) == 0]

print("Books with empty genre lists:", len(empty_genres))

# Display the affected books
empty_genres[["Book", "Genres", "Description"]]

Books with empty genre lists: 0


,Book,Genres,Description


In [136]:
# Keep books that have at least one genre
df_labeled = df_clean[
    df_clean["Genre_List"].apply(len) > 0
].copy()

# Extract the first listed genre
df_labeled["Primary_Genre"] = df_labeled["Genre_List"].apply(
    lambda genres: genres[0]
)

print("Original number of books:", len(df_clean))
print("Books with genre labels:", len(df_labeled))
print("Books excluded:", len(df_clean) - len(df_labeled))

Original number of books: 9040
Books with genre labels: 9040
Books excluded: 0


In [137]:
print(
    "Books with missing descriptions:",
    df_labeled["Description"].isna().sum()
)

print(
    "Books with usable descriptions:",
    df_labeled["Description"].notna().sum()
)

Books with missing descriptions: 40
Books with usable descriptions: 9000


In [138]:
# Remove books that do not have descriptions
df_model = df_labeled.dropna(subset=["Description"]).copy()

print("Number of books for modeling:", len(df_model))
print("Missing descriptions:", df_model["Description"].isna().sum())
print("Missing genre labels:", df_model["Primary_Genre"].isna().sum())

Number of books for modeling: 9000
Missing descriptions: 0
Missing genre labels: 0


In [139]:
# Exclude the Genre_List column because it contains lists
duplicate_columns = [
    col for col in df_model.columns
    if col != "Genre_List"
]

print(
    "Exact duplicate rows:",
    df_model.duplicated(subset=duplicate_columns).sum()
)

print(
    "Duplicate Book/Author combinations:",
    df_model.duplicated(subset=["Book", "Author"]).sum()
)

Exact duplicate rows: 0
Duplicate Book/Author combinations: 101


In [140]:
duplicate_books = df_model[
    df_model.duplicated(
        subset=["Book", "Author"],
        keep=False
    )
].sort_values(
    by=["Book", "Author"]
)

duplicate_books[
    ["Book", "Author", "Description", "Primary_Genre", "URL"]
].head(30)

,Book,Author,Description,Primary_Genre,URL
613,'Salem's Lot,Stephen King,Librarian's Note: Alternate-cover edition for ...,Horror,https://www.goodreads.com/book/show/11590._Sal...
4605,'Salem's Lot,Stephen King,"Also contains: One For the Road, Jerusalem's L...",Horror,https://www.goodreads.com/book/show/5413._Sale...
1115,A Castle of Doomsday,Michael G. Kramer,"A Castle of Doomsday, is historical fiction cl...",Historical Fiction,https://www.goodreads.com/book/show/55476672-a...
2026,A Castle of Doomsday,Michael G. Kramer,A Journey through English history from Edward ...,Historical Fiction,https://www.goodreads.com/book/show/74917267-a...
2609,A Court of Mist and Fury (A Court of Thorns an...,Sarah J. Maas,Feyre survived Amarantha's clutches to return ...,Fantasy,https://www.goodreads.com/book/show/17927395-a...
4769,A Court of Mist and Fury (A Court of Thorns an...,Sarah J. Maas,The seductive and stunning #1 New York Times b...,Fantasy,https://www.goodreads.com/book/show/50659468-a...
1786,A Gentleman in Moscow,Amor Towles,A Gentleman in Moscow immerses us in another e...,Historical Fiction,https://www.goodreads.com/book/show/29430012-a...
3086,A Gentleman in Moscow,Amor Towles,From the New York Times bestselling author of ...,Historical Fiction,https://www.goodreads.com/book/show/34066798-a...
704,A Man Called Ove,Fredrik Backman,A grumpy yet loveable man finds his solitary w...,Fiction,https://www.goodreads.com/book/show/18774964-a...
8243,A Man Called Ove,Fredrik Backman,An alternate cover edition for this ASIN can b...,Fiction,https://www.goodreads.com/book/show/20492604-a...


In [141]:
# Count descriptions that are missing
print(
    "Missing descriptions:",
    df_clean["Description"].isna().sum()
)

# Count descriptions that are empty or contain only spaces
blank_descriptions = (
    df_clean["Description"].notna()
    & df_clean["Description"].str.strip().eq("")
)

print(
    "Empty descriptions:",
    blank_descriptions.sum()
)

# Remove books with missing descriptions
df_model = df_clean.dropna(
    subset=["Description"]
).copy()

# Remove leading and trailing whitespace
df_model["Description"] = df_model["Description"].str.strip()

# Remove descriptions that are completely empty
df_model = df_model[
    df_model["Description"] != ""
].copy()

print("Final number of books:", len(df_model))

Missing descriptions: 40
Empty descriptions: 0
Final number of books: 9000


In [142]:
# Count the number of books in each primary genre
genre_counts = df_model["Primary_Genre"].value_counts()

# Display the 20 most common genres
print(genre_counts.head(20))

# Display the 20 least common genres
print(genre_counts.tail(20))

# Count the number of unique genres
print(
    "Number of unique genres:",
    df_model["Primary_Genre"].nunique()
)

# Display the smallest genre size
print(
    "Smallest genre size:",
    genre_counts.min()
)

Primary_Genre
Fiction               1706
Nonfiction             953
Fantasy                939
Classics               515
Historical Fiction     467
Mystery                387
Young Adult            386
Science Fiction        325
Romance                297
History                197
Horror                 192
Thriller               178
Self Help              153
Poetry                 146
Philosophy             139
Business               112
Plays                  106
Childrens               92
Picture Books           89
Psychology              85
Name: count, dtype: int64
Primary_Genre
Angels                 1
Martial Arts           1
Judaism                1
Amish                  1
Biography Memoir       1
Adoption               1
Light Novel            1
Juvenile               1
Love Inspired          1
Shapeshifters          1
Canada                 1
Cultural               1
Cooking                1
Fae                    1
Horses                 1
Womens                 1
Milita

In [143]:
# Count how many genres have different numbers of books
print("Genres with fewer than 3 books:",
      (genre_counts < 3).sum())

print("Genres with fewer than 5 books:",
      (genre_counts < 5).sum())

print("Genres with fewer than 10 books:",
      (genre_counts < 10).sum())

print("Genres with fewer than 20 books:",
      (genre_counts < 20).sum())

Genres with fewer than 3 books: 103
Genres with fewer than 5 books: 118
Genres with fewer than 10 books: 145
Genres with fewer than 20 books: 163


In [144]:
# Display genres with fewer than 8 books
rare_genres = genre_counts[genre_counts < 8]

print(rare_genres)
print("Total books in these rare genres:", rare_genres.sum())

Primary_Genre
Cookbooks               7
Time Travel             7
Mythology               7
Contemporary Romance    7
Africa                  6
                       ..
Womens                  1
Military History        1
Wicca                   1
Romanian Literature     1
Civil War               1
Name: count, Length: 137, dtype: int64
Total books in these rare genres: 289


In [145]:
# Set the minimum number of books required for a genre
minimum_books = 8

# Count the number of books in each primary genre
genre_counts = df_model["Primary_Genre"].value_counts()

# Keep genres with at least 8 books then groups the rest into "Other"
df_model["Final_Genre"] = df_model["Primary_Genre"].apply(
    lambda genre: genre if genre_counts[genre] >= minimum_books else "Other"
)

# Display the new genre counts
final_genre_counts = df_model["Final_Genre"].value_counts()

print("Number of genre labels:", len(final_genre_counts))
print("\nGenre counts:")
print(final_genre_counts)

Number of genre labels: 70

Genre counts:
Final_Genre
Fiction               1706
Nonfiction             953
Fantasy                939
Classics               515
Historical Fiction     467
                      ... 
Animals                  9
Writing                  8
Inspirational            8
Erotica                  8
LGBT                     8
Name: count, Length: 70, dtype: int64


In [146]:
# Group books by title and author
duplicate_group_summary = df_model.groupby(
    ["Book", "Author"]
).agg(
    Number_of_Rows=("Final_Genre", "size"),
    Number_of_Genres=("Final_Genre", "nunique")
)

# Keep only book-author combinations appearing more than once
repeated_groups = duplicate_group_summary[
    duplicate_group_summary["Number_of_Rows"] > 1
]

# Find repeated combinations with different primary genres
conflicting_groups = repeated_groups[
    repeated_groups["Number_of_Genres"] > 1
]

print("Repeated Book/Author combinations:", len(repeated_groups))
print("Repeated combinations with different genres:", len(conflicting_groups))

# Display a few examples
print("\nExamples of conflicting combinations:")
print(conflicting_groups.head(10))

Repeated Book/Author combinations: 91
Repeated combinations with different genres: 0

Examples of conflicting combinations:
Empty DataFrame
Columns: [Number_of_Rows, Number_of_Genres]
Index: []


In [147]:
# Identify every row belonging to a repeated book-author combination
repeated_rows = df_model.duplicated(
    subset=["Book", "Author"],
    keep=False
)

# Count rows belonging to repeated combinations
print("Rows belonging to repeated combinations:", repeated_rows.sum())

# Count rows belonging to unique combinations
print("Rows belonging to unique combinations:", (~repeated_rows).sum())

Rows belonging to repeated combinations: 192
Rows belonging to unique combinations: 8808


In [148]:
# Assign a group ID to each unique book-author combination
groups = df_model.groupby(
    ["Book", "Author"],
    sort=False
).ngroup()

# Check the number of unique groups
print("Number of book-author groups:", groups.nunique())

Number of book-author groups: 8899


In [149]:
# Create a splitter that divides the data into 10 groups
test_splitter = StratifiedGroupKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

# Generate the training/validation and testing indices
train_val_indices, test_indices = next(
    test_splitter.split(
        df_model,
        df_model["Final_Genre"],
        groups
    )
)

# Separate the training/validation and testing data
train_val_df = df_model.iloc[train_val_indices].copy()
test_df = df_model.iloc[test_indices].copy()

# Create a second splitter for the validation set
validation_splitter = StratifiedGroupKFold(
    n_splits=6,
    shuffle=True,
    random_state=42
)

# Generate the training and validation indices
train_indices, validation_indices = next(
    validation_splitter.split(
        train_val_df,
        train_val_df["Final_Genre"],
        groups.iloc[train_val_indices]
    )
)

# Create the final three datasets
train_df = train_val_df.iloc[train_indices].copy()
validation_df = train_val_df.iloc[validation_indices].copy()

# Display the dataset sizes
print("Training entries:", len(train_df))
print("Validation entries:", len(validation_df))
print("Testing entries:", len(test_df))

c:\Users\nelso\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_split.py:1036: UserWarning: The least populated class in y has only 8 members, which is less than n_splits=10.
  warnings.warn(


Training entries: 6751
Validation entries: 1349
Testing entries: 900


In [150]:
# Get a list of all 70 genres
all_genres = sorted(df_model["Final_Genre"].unique())

# Check each dataset
for name, dataset in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Testing", test_df)
]:
    # Count how many books belong to each genre
    genre_counts = dataset["Final_Genre"].value_counts()

    # Find genres that have zero books in this dataset
    missing_genres = [
        genre for genre in all_genres
        if genre_counts.get(genre, 0) == 0
    ]

    print(f"\n{name} dataset:")
    print("Number of represented genres:", len(all_genres) - len(missing_genres))
    print("Missing genres:", missing_genres)


Training dataset:
Number of represented genres: 70
Missing genres: []

Validation dataset:
Number of represented genres: 70
Missing genres: []

Testing dataset:
Number of represented genres: 68
Missing genres: ['Erotica', 'Writing']
